# Stage 4

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
df_general = pd.read_csv('D:/Rakamin/Finpro//dataset/general_data.csv')
df_employee = pd.read_csv('D:/Rakamin/Finpro/dataset/employee_survey_data.csv')
df_manager = pd.read_csv('D:/Rakamin/Finpro//dataset/manager_survey_data.csv')
df_in_time = pd.read_csv('D:/Rakamin/Finpro//dataset/in_time.csv')
df_out_time = pd.read_csv('D:/Rakamin/Finpro//dataset/out_time.csv')

In [4]:
df_manager.columns

Index(['EmployeeID', 'JobInvolvement', 'PerformanceRating'], dtype='str')

In [5]:
set(df_general['BusinessTravel'])

{'Non-Travel', 'Travel_Frequently', 'Travel_Rarely'}

In [6]:
cols_to_fix = df_in_time.columns[1:]

df_in_time[cols_to_fix] = df_in_time[cols_to_fix].apply(pd.to_datetime)
df_out_time[cols_to_fix] = df_out_time[cols_to_fix].apply(pd.to_datetime)
work_hours = df_out_time.iloc[:, 1:] - df_in_time.iloc[:, 1:]

In [7]:
work_hours = (df_out_time.iloc[:, 1:] - df_in_time.iloc[:, 1:]) / pd.Timedelta(hours=1)
total_work_hours = work_hours.sum(axis=1)
average_work_hours = work_hours.mean(axis=1)

df_in_time.iloc[:, 1:] = df_in_time.iloc[:, 1:].apply(pd.to_datetime, errors='coerce')
df_out_time.iloc[:, 1:] = df_out_time.iloc[:, 1:].apply(pd.to_datetime, errors='coerce')

in_long = df_in_time.melt(id_vars=['Unnamed: 0'],
                     var_name='date',
                     value_name='in_time')

out_long = df_out_time.melt(id_vars=['Unnamed: 0'],
                       var_name='date',
                       value_name='out_time')

df = in_long.merge(out_long, on=['Unnamed: 0', 'date'])

df = df.dropna(subset=['in_time', 'out_time'])

df['work_hours'] = (df['out_time'] - df['in_time']).dt.total_seconds() / 3600

df = df[df['work_hours'] >= 0]

df['is_overwork'] = df['work_hours'] >= 9

overwork_days = df.groupby('Unnamed: 0')['is_overwork'].sum()
overwork_days = overwork_days.reset_index(drop=True)

In [8]:
df_in_out = pd.concat(
    [total_work_hours, average_work_hours, overwork_days],
    axis=1
)

df_in_out.columns = ['total_work_hours', 'average_work_hours', 'overwork_days']
df_in_out = df_in_out.reset_index()

In [9]:
df_in_out['overwork'] = df_in_out['total_work_hours'].apply(
    lambda x: 'Yes' if x > 2080 else 'No'
)

In [10]:
df_in_out = df_in_out.drop(columns=['index'])

df_in_out.insert(0, 'EmployeeID', range(1, len(df_in_out) + 1))

In [11]:
final_df = pd.merge(df_general, df_employee, on='EmployeeID')
final_df = pd.merge(final_df,df_manager, on='EmployeeID')
final_df = pd.merge(final_df,df_in_out, on='EmployeeID')

In [12]:
final_df['Attrition'] = final_df['Attrition'].map({'Yes': 1, 'No': 0})
final_df['isMale'] = final_df['Gender'].map({'Male': 1, 'Female': 0})
final_df['overwork'] = final_df['overwork'].map({'Yes': 1, 'No': 0})

In [13]:
final_df = pd.get_dummies(final_df, columns=['Department'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['EducationField'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['JobRole'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['MaritalStatus'],dtype=int, drop_first=True)
final_df = pd.get_dummies(final_df, columns=['BusinessTravel'],dtype=int, drop_first=True)

In [14]:
pd.set_option('display.max_columns', None)
final_df.head()

,Age,Attrition,DistanceFromHome,Education,EmployeeCount,EmployeeID,Gender,JobLevel,MonthlyIncome,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating,total_work_hours,average_work_hours,overwork_days,overwork,isMale,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely
0,51,0,6,2,1,1,Female,1,131160,1.0,Y,11,8,0,1.0,6,1,0,0,3.0,4.0,2.0,3,3,1710.686944,7.373651,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
1,31,1,10,1,1,2,Female,1,41890,0.0,Y,23,8,1,6.0,3,5,1,4,3.0,2.0,4.0,2,4,1821.676667,7.718969,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0
2,32,0,17,4,1,3,Male,4,193280,1.0,Y,15,8,3,5.0,2,5,0,3,2.0,2.0,1.0,3,3,1697.204167,7.013240,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0
3,38,0,2,5,1,4,Male,3,83210,3.0,Y,11,8,3,13.0,5,8,7,5,4.0,4.0,3.0,2,3,1690.514444,7.193678,0,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
4,32,0,10,1,1,5,Male,1,23420,4.0,Y,12,8,2,9.0,2,6,0,4,4.0,1.0,3.0,3,3,1961.512778,8.006175,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,1


## Split Dataset & Preprocessing

In [15]:
from sklearn.model_selection import train_test_split

y = final_df['Attrition']
X = final_df.drop(columns=['Attrition'])

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.4,
    random_state= 42,
    stratify=y
)

In [16]:
X_train.fillna({'EnvironmentSatisfaction':X_train['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X_train.fillna({'JobSatisfaction':X_train['JobSatisfaction'].mode()[0]}, inplace=True)
X_train.fillna({'WorkLifeBalance':X_train['WorkLifeBalance'].mode()[0]}, inplace=True)
X_train.fillna({'NumCompaniesWorked':X_train['NumCompaniesWorked'].mode()[0]}, inplace=True)
X_train.fillna({'TotalWorkingYears':X_train['TotalWorkingYears'].mode()[0]}, inplace=True)

X_val.fillna({'EnvironmentSatisfaction':X_val['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X_val.fillna({'JobSatisfaction':X_val['JobSatisfaction'].mode()[0]}, inplace=True)
X_val.fillna({'WorkLifeBalance':X_val['WorkLifeBalance'].mode()[0]}, inplace=True)
X_val.fillna({'NumCompaniesWorked':X_val['NumCompaniesWorked'].mode()[0]}, inplace=True)
X_val.fillna({'TotalWorkingYears':X_val['TotalWorkingYears'].mode()[0]}, inplace=True)

X.fillna({'EnvironmentSatisfaction':X['EnvironmentSatisfaction'].mode()[0]}, inplace=True)
X.fillna({'JobSatisfaction':X['JobSatisfaction'].mode()[0]}, inplace=True)
X.fillna({'WorkLifeBalance':X['WorkLifeBalance'].mode()[0]}, inplace=True)
X.fillna({'NumCompaniesWorked':X['NumCompaniesWorked'].mode()[0]}, inplace=True)
X.fillna({'TotalWorkingYears':X['TotalWorkingYears'].mode()[0]}, inplace=True)

,Age,DistanceFromHome,Education,EmployeeCount,EmployeeID,Gender,JobLevel,MonthlyIncome,NumCompaniesWorked,Over18,PercentSalaryHike,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,YearsAtCompany,YearsSinceLastPromotion,YearsWithCurrManager,EnvironmentSatisfaction,JobSatisfaction,WorkLifeBalance,JobInvolvement,PerformanceRating,total_work_hours,average_work_hours,overwork_days,overwork,isMale,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely
0,51,6,2,1,1,Female,1,131160,1.0,Y,11,8,0,1.0,6,1,0,0,3.0,4.0,2.0,3,3,1710.686944,7.373651,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
1,31,10,1,1,2,Female,1,41890,0.0,Y,23,8,1,6.0,3,5,1,4,3.0,2.0,4.0,2,4,1821.676667,7.718969,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0
2,32,17,4,1,3,Male,4,193280,1.0,Y,15,8,3,5.0,2,5,0,3,2.0,2.0,1.0,3,3,1697.204167,7.013240,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0
3,38,2,5,1,4,Male,3,83210,3.0,Y,11,8,3,13.0,5,8,7,5,4.0,4.0,3.0,2,3,1690.514444,7.193678,0,0,1,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0
4,32,10,1,1,5,Male,1,23420,4.0,Y,12,8,2,9.0,2,6,0,4,4.0,1.0,3.0,3,3,1961.512778,8.006175,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4405,42,5,4,1,4406,Female,1,60290,3.0,Y,17,8,1,10.0,5,3,0,2,4.0,1.0,3.0,3,3,2070.913333,8.522277,18,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,1
4406,29,2,4,1,4407,Male,1,26790,2.0,Y,15,8,0,10.0,2,3,0,2,4.0,4.0,3.0,2,3,1468.401944,6.092954,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1
4407,25,25,2,1,4408,Male,2,37020,0.0,Y,20,8,0,5.0,4,4,1,2,1.0,3.0,3.0,3,4,1780.231944,7.706632,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1
4408,42,18,2,1,4409,Male,1,23980,0.0,Y,14,8,1,10.0,2,9,7,8,4.0,1.0,3.0,2,3,2287.715278,9.492595,223,1,1,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1


In [17]:
X_train['DistanceFromHome_log'] = np.log1p(X_train['DistanceFromHome'])
X_train['MonthlyIncome_log'] = np.log1p(X_train['MonthlyIncome'])
X_train['NumCompaniesWorked_log'] = np.log1p(X_train['NumCompaniesWorked'])
X_train['PercentSalaryHike_log'] = np.log1p(X_train['PercentSalaryHike'])
X_train['TotalWorkingYears_log'] = np.log1p(X_train['TotalWorkingYears'])
X_train['YearsAtCompany_log'] = np.log1p(X_train['YearsAtCompany'])
X_train['YearsSinceLastPromotion_log'] = np.log1p(X_train['YearsSinceLastPromotion'])
X_train['YearsWithCurrManager_log'] = np.log1p(X_train['YearsWithCurrManager'])

X_val['DistanceFromHome_log'] = np.log1p(X_val['DistanceFromHome'])
X_val['MonthlyIncome_log'] = np.log1p(X_val['MonthlyIncome'])
X_val['NumCompaniesWorked_log'] = np.log1p(X_val['NumCompaniesWorked'])
X_val['PercentSalaryHike_log'] = np.log1p(X_val['PercentSalaryHike'])
X_val['TotalWorkingYears_log'] = np.log1p(X_val['TotalWorkingYears'])
X_val['YearsAtCompany_log'] = np.log1p(X_val['YearsAtCompany'])
X_val['YearsSinceLastPromotion_log'] = np.log1p(X_val['YearsSinceLastPromotion'])
X_val['YearsWithCurrManager_log'] = np.log1p(X_val['YearsWithCurrManager'])

X['DistanceFromHome_log'] = np.log1p(X['DistanceFromHome'])
X['MonthlyIncome_log'] = np.log1p(X['MonthlyIncome'])
X['NumCompaniesWorked_log'] = np.log1p(X['NumCompaniesWorked'])
X['PercentSalaryHike_log'] = np.log1p(X['PercentSalaryHike'])
X['TotalWorkingYears_log'] = np.log1p(X['TotalWorkingYears'])
X['YearsAtCompany_log'] = np.log1p(X['YearsAtCompany'])
X['YearsSinceLastPromotion_log'] = np.log1p(X['YearsSinceLastPromotion'])
X['YearsWithCurrManager_log'] = np.log1p(X['YearsWithCurrManager'])

In [18]:
X_train = X_train.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])
X_val = X_val.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])
X = X.drop(columns= ['EmployeeID','DistanceFromHome','Gender','MonthlyIncome','NumCompaniesWorked','PercentSalaryHike',
                                   'TotalWorkingYears','YearsAtCompany','YearsAtCompany','YearsSinceLastPromotion','YearsWithCurrManager',
                                   'TotalWorkingYears_log','YearsSinceLastPromotion_log','YearsWithCurrManager_log',
                                   'average_work_hours','overwork_days','overwork','StandardHours','EmployeeCount','Over18'])

In [19]:
X_train.columns

Index(['Age', 'Education', 'JobLevel', 'StockOptionLevel',
       'TrainingTimesLastYear', 'EnvironmentSatisfaction', 'JobSatisfaction',
       'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating',
       'total_work_hours', 'isMale', 'Department_Research & Development',
       'Department_Sales', 'EducationField_Life Sciences',
       'EducationField_Marketing', 'EducationField_Medical',
       'EducationField_Other', 'EducationField_Technical Degree',
       'JobRole_Human Resources', 'JobRole_Laboratory Technician',
       'JobRole_Manager', 'JobRole_Manufacturing Director',
       'JobRole_Research Director', 'JobRole_Research Scientist',
       'JobRole_Sales Executive', 'JobRole_Sales Representative',
       'MaritalStatus_Married', 'MaritalStatus_Single',
       'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely',
       'DistanceFromHome_log', 'MonthlyIncome_log', 'NumCompaniesWorked_log',
       'PercentSalaryHike_log', 'YearsAtCompany_log'],
      dtype='st

## Modeling

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import cross_validate

def eval_classification(model):
    # prediction
    y_pred = model.predict(X_val)
    y_pred_train = model.predict(X_train)

    # probability
    y_proba = model.predict_proba(X_val)[:,1]
    y_proba_train = model.predict_proba(X_train)[:,1]

    # metrics
    print("Accuracy (val Set): %.2f" % accuracy_score(y_val, y_pred))
    print("Accuracy (Train Set): %.2f" % accuracy_score(y_train, y_pred_train))
    print("Precision (val Set): %.2f" % precision_score(y_val, y_pred))
    print("Recall (val Set): %.2f" % recall_score(y_val, y_pred))
    print("F1-Score (val Set): %.2f" % f1_score(y_val, y_pred))
    print("roc_auc (val): %.2f" % roc_auc_score(y_val, y_proba))
    print("roc_auc (train): %.2f" % roc_auc_score(y_train, y_proba_train))

    score = cross_validate(model, X, y, cv=5, scoring='recall', return_train_score=True)
    print('recall (cv train):', score['train_score'].mean())
    print('recall (cv test):', score['test_score'].mean())

def show_feature_importance(model):
    feat_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    ax = feat_importances.nlargest(25).plot(kind='barh', figsize=(10, 8))
    ax.invert_yaxis()

    plt.xlabel('score')
    plt.ylabel('feature')
    plt.title('feature importance score')

def show_best_hyperparameter(model):
    print(model.best_estimator_.get_params())

### CatBoost

In [22]:
from catboost import CatBoostClassifier
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# pipeline
pipeline_catboost = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier(
        random_state=42,
        verbose=0
    ))
])

# train model
pipeline_catboost.fit(X_train, y_train)

# prediction
y_pred = pipeline_catboost.predict(X_val)

# evaluation
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98      1480
           1       0.97      0.77      0.86       284

    accuracy                           0.96      1764
   macro avg       0.97      0.88      0.92      1764
weighted avg       0.96      0.96      0.96      1764



In [23]:
eval_classification(pipeline_catboost)

Accuracy (val Set): 0.96
Accuracy (Train Set): 1.00
Precision (val Set): 0.97
Recall (val Set): 0.77
F1-Score (val Set): 0.86
roc_auc (val): 0.96
roc_auc (train): 1.00
recall (cv train): 0.9799574246887299
recall (cv test): 0.9507830197971042


In [24]:
print(confusion_matrix(y_val, pipeline_catboost.predict(X_val)))

[[1474    6]
 [  65  219]]


#### Paramater Tuning

In [25]:
# pipeline
pipeline_catboost = Pipeline([
    ('scaler', StandardScaler()),
    ('model', CatBoostClassifier(
        random_state=42,
        verbose=0
    ))
])

# hyperparameter tuning
param_catboost = {
    'model__iterations': [100, 200, 300, 500],
    'model__depth': [4, 6, 8, 10],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__l2_leaf_reg': [1, 3, 5, 7, 9],
    'model__border_count': [32, 64, 128, 255],
    'model__bagging_temperature': [0, 1, 3, 5],
    'model__random_strength': [1, 2, 5, 10]
}

# randomized search
search_catboost = RandomizedSearchCV(
    estimator=pipeline_catboost,
    param_distributions=param_catboost,
    n_iter=20,
    scoring='recall',   # focus on increasing recall
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# training
search_catboost.fit(X_train, y_train)

# best model
best_catboost = search_catboost.best_estimator_

# prediction
y_pred = best_catboost.predict(X_val)

# evaluation
print("Best Parameters:")
print(search_catboost.best_params_)

print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters:
{'model__random_strength': 5, 'model__learning_rate': 0.2, 'model__l2_leaf_reg': 3, 'model__iterations': 500, 'model__depth': 4, 'model__border_count': 32, 'model__bagging_temperature': 1}

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1480
           1       0.96      0.85      0.90       284

    accuracy                           0.97      1764
   macro avg       0.97      0.92      0.94      1764
weighted avg       0.97      0.97      0.97      1764



In [26]:
eval_classification(search_catboost)

Accuracy (val Set): 0.97
Accuracy (Train Set): 1.00
Precision (val Set): 0.96
Recall (val Set): 0.85
F1-Score (val Set): 0.90
roc_auc (val): 0.95
roc_auc (train): 1.00
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Fitting 5 folds for each of 20 candidates, totalling 100 fits
recall (cv train): 1.0
recall (cv test): 0.9985915492957746


In [27]:
print(confusion_matrix(y_val, search_catboost.predict(X_val)))

[[1470   10]
 [  42  242]]


#### Tuning Threshold

In [48]:
y_prob = best_catboost.predict_proba(X_val)[:, 1]
y_pred_t = (y_prob >= 0.5).astype(int)
roc_auc = roc_auc_score(y_val, y_prob)
print(classification_report(y_val, y_pred_t))
print(confusion_matrix(y_val, y_pred_t))
print(f"ROC AUC Score: {roc_auc:.2f}")

              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1480
           1       0.96      0.85      0.90       284

    accuracy                           0.97      1764
   macro avg       0.97      0.92      0.94      1764
weighted avg       0.97      0.97      0.97      1764

[[1470   10]
 [  42  242]]
ROC AUC Score: 0.95


In [53]:
y_prob = best_catboost.predict_proba(X_val)[:, 1]
y_pred_t = (y_prob >= 0.1).astype(int)
roc_auc = roc_auc_score(y_val, y_prob)
print(classification_report(y_val, y_pred_t))
print(confusion_matrix(y_val, y_pred_t))
print(f"ROC AUC Score: {roc_auc:.2f}")

              precision    recall  f1-score   support

           0       0.98      0.97      0.97      1480
           1       0.85      0.88      0.87       284

    accuracy                           0.96      1764
   macro avg       0.92      0.93      0.92      1764
weighted avg       0.96      0.96      0.96      1764

[[1437   43]
 [  34  250]]
ROC AUC Score: 0.95


In [50]:
y_prob = best_catboost.predict_proba(X_val)[:, 1]
y_pred_t = (y_prob >= 0.75).astype(int)
roc_auc = roc_auc_score(y_val, y_prob)
print(classification_report(y_val, y_pred_t))
print(confusion_matrix(y_val, y_pred_t))
print(f"ROC AUC Score: {roc_auc:.2f}")

              precision    recall  f1-score   support

           0       0.96      0.99      0.98      1480
           1       0.97      0.81      0.88       284

    accuracy                           0.96      1764
   macro avg       0.97      0.90      0.93      1764
weighted avg       0.96      0.96      0.96      1764

[[1472    8]
 [  55  229]]
ROC AUC Score: 0.95


In [54]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

# 1. DEF INISIKAN ASUMSI BIAYA PERUSAHAAN (Dua angka ini bisa Anda ganti)
COST_FP = 10000000  # Rp 10 Juta (Biaya program retensi per orang)
COST_FN = 60000000  # Rp 60 Juta (Biaya rekrutmen & kehilangan produktivitas per orang)

# 2. SIMULASI DATA (Ganti y_val dan y_prob dengan variabel milik Anda)
# y_val = data target aktual Anda
# y_prob = best_catboost.predict_proba(X_val)[:, 1]

# (Contoh dummy agar code bisa dijalankan secara independen)
np.random.seed(42)
y_val = np.array([0]*1512 + [1]*284) # Total 1796 data seperti di image Anda
y_prob = np.random.uniform(0, 1, size=len(y_val)) 

# 3. PROSES PENCARIAN THRESHOLD OPTIMAL
list_thresholds = np.linspace(0.01, 0.99, 99) # Menguji dari 0.01 sampai 0.99
results = []

for t in list_thresholds:
    # Bikin prediksi berdasarkan threshold t
    y_pred_t = (y_prob >= t).astype(int)
    
    # Ambil nilai confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred_t).ravel()
    
    # Hitung total kerugian finansial akibat error model
    total_cost = (fp * COST_FP) + (fn * COST_FN)
    
    results.append({
        'Threshold': round(t, 2),
        'False Positive': fp,
        'False Negative': fn,
        'Total Loss (IDR)': total_cost
    })

# Convert ke DataFrame untuk mempermudah visualisasi data
df_res = pd.DataFrame(results)

# 4. CARI THRESHOLD DENGAN KERUGIAN TERENDAH
best_row = df_res.loc[df_res['Total Loss (IDR)'].idxmin()]

print("==================================================")
print("       HASIL OPTIMASI THRESHOLD BERDASARKAN BIAYA   ")
print("==================================================")
print(f"Threshold Terbaik Secara Bisnis : {best_row['Threshold']}")
print(f"Prediksi False Positive (FP)    : {int(best_row['False Positive'])} orang")
print(f"Prediksi False Negative (FN)    : {int(best_row['False Negative'])} orang")
print(f"Minimum Total Kerugian Finansial: Rp {best_row['Total Loss (IDR)']:,.0f}")
print("==================================================")

       HASIL OPTIMASI THRESHOLD BERDASARKAN BIAYA   
Threshold Terbaik Secara Bisnis : 0.05
Prediksi False Positive (FP)    : 1428 orang
Prediksi False Negative (FN)    : 11 orang
Minimum Total Kerugian Finansial: Rp 14,940,000,000


## Deployment

In [28]:
import joblib
from sklearn.preprocessing import OneHotEncoder

joblib.dump(search_catboost, 'my_model.joblib')

['my_model.joblib']

In [29]:
import joblib
X_columns = X_train.columns
joblib.dump(X_columns, "columns.pkl")

['columns.pkl']